# Apply Glossary Terms to Census Bureau ACS Dataset

This notebook demonstrates how to create a Dataplex Business Glossary following ISO 11179 principles by establishing Object Classes (categories) for ACS demographics data and linking glossary terms to specific columns in census tables.

## What This Notebook Does

1. **Analyzes table schema** - Queries BigQuery to get all columns from `blockgroup_2018_5yr`
2. **Creates business glossary** - Establishes a root glossary for ACS Demographics
3. **Creates categories** - Implements ISO 11179 Object Classes as Dataplex categories:
   - Age and Sex
   - Race and Ethnicity
   - Commuting Characteristics
   - Labor Force
   - Educational Attainment
4. **Creates glossary terms** - Generates business terms for each categorized column
5. **Links terms to data assets** - Connects glossary terms to BigQuery table columns
6. **Validates and explores** - Verifies the glossary structure and provides exploration links

## ISO 11179 Alignment

This implementation follows ISO 11179 metadata registry standards:
- **Object Class**: Represented as Dataplex Glossary Categories (e.g., "Age and Sex")
- **Data Elements**: Represented as Glossary Terms linked to actual table columns
- **Classification Scheme**: Hierarchical category structure for organizing demographic concepts

## Prerequisites

- Census dataset already created (via `config_and_data_setup.ipynb`)
- Required IAM roles:
  - `roles/dataplex.catalogAdmin` (to create glossary and terms)
  - `roles/bigquery.admin` (to query table schemas)

## Estimated Time

15-25 minutes (depending on number of terms created)

---
# Section 1: Configuration & Authentication

Configure project settings and verify authentication.

In [32]:
# Configuration variables
# ⚠️  IMPORTANT: Use the same PROJECT_ID and DATASET_ID from config_and_data_setup.ipynb

PROJECT_ID = "johnswain-1200-20260303091357"  # Replace with your project ID (same as config_and_data_setup.ipynb)
REGION = "us-central1"                         # GCP region
CATALOG_LOCATION = "us"                        # Catalog location (must match BigQuery dataset location)
DATASET_ID = "census_bureau_acs"              # BigQuery dataset name (same as config_and_data_setup.ipynb)
TABLE_ID = "blockgroup_2018_5yr"              # Target table for glossary application
GLOSSARY_ID = "acs-demographics-glossary"     # ID for the glossary to create

print("📋 Configuration:")
print(f"  Project ID: {PROJECT_ID}")
print(f"  Region: {REGION}")
print(f"  Catalog Location: {CATALOG_LOCATION}")
print(f"  Dataset: {DATASET_ID}")
print(f"  Table: {TABLE_ID}")
print(f"  Glossary ID: {GLOSSARY_ID}")
print()
print("✅ Configuration loaded")
print()
print("💡 Tip: This should match the PROJECT_ID and DATASET_ID from config_and_data_setup.ipynb")

📋 Configuration:
  Project ID: johnswain-1200-20260303091357
  Region: us-central1
  Catalog Location: us
  Dataset: census_bureau_acs
  Table: blockgroup_2018_5yr
  Glossary ID: acs-demographics-glossary

✅ Configuration loaded

💡 Tip: This should match the PROJECT_ID and DATASET_ID from config_and_data_setup.ipynb


In [33]:
# Verify authentication
from google.auth import default
import google.auth

try:
    credentials, project = default()
    print("🔐 Authentication Status:")
    print(f"  ✅ Authenticated successfully")
    print(f"  ✅ Default project: {project}")
    
    if project != PROJECT_ID:
        print(f"  ⚠️  Note: Using PROJECT_ID ({PROJECT_ID}) instead of default ({project})")
    
except Exception as e:
    print(f"❌ Authentication failed: {e}")
    print("Please run: gcloud auth application-default login")
    raise

🔐 Authentication Status:
  ✅ Authenticated successfully
  ✅ Default project: graph-demo-471710
  ⚠️  Note: Using PROJECT_ID (johnswain-1200-20260303091357) instead of default (graph-demo-471710)


In [34]:
# Check required IAM permissions
print("🔐 Required IAM Permissions:")
print()
print("This notebook requires the following permissions:")
print()
print("   1️⃣  Dataplex Catalog:")
print("      - roles/dataplex.catalogAdmin (to create glossary, categories, and terms)")
print()
print("   2️⃣  BigQuery:")
print("      - roles/bigquery.admin (to query table schemas)")
print()
print("💡 To grant these permissions, run these commands in your terminal:")
print()
print(f"   # Grant Dataplex Catalog Admin")
print(f"   gcloud projects add-iam-policy-binding {PROJECT_ID} \\")
print(f"     --member='user:YOUR_EMAIL' \\")
print(f"     --role='roles/dataplex.catalogAdmin'")
print()
print(f"   # Grant BigQuery Admin")
print(f"   gcloud projects add-iam-policy-binding {PROJECT_ID} \\")
print(f"     --member='user:YOUR_EMAIL' \\")
print(f"     --role='roles/bigquery.admin'")
print()
print("   (Replace YOUR_EMAIL with your Google Cloud account email)")
print()
print("⏱️  After granting permissions, wait 1-2 minutes for IAM propagation")

🔐 Required IAM Permissions:

This notebook requires the following permissions:

   1️⃣  Dataplex Catalog:
      - roles/dataplex.catalogAdmin (to create glossary, categories, and terms)

   2️⃣  BigQuery:
      - roles/bigquery.admin (to query table schemas)

💡 To grant these permissions, run these commands in your terminal:

   # Grant Dataplex Catalog Admin
   gcloud projects add-iam-policy-binding johnswain-1200-20260303091357 \
     --member='user:YOUR_EMAIL' \
     --role='roles/dataplex.catalogAdmin'

   # Grant BigQuery Admin
   gcloud projects add-iam-policy-binding johnswain-1200-20260303091357 \
     --member='user:YOUR_EMAIL' \
     --role='roles/bigquery.admin'

   (Replace YOUR_EMAIL with your Google Cloud account email)

⏱️  After granting permissions, wait 1-2 minutes for IAM propagation


In [35]:
# Install required libraries
import sys
import subprocess

print("📦 Installing required libraries...")
print()

libraries = [
    "google-cloud-dataplex",
    "google-cloud-bigquery"
]

for lib in libraries:
    print(f"   Installing {lib}...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", lib])

print()
print("✅ All libraries installed")
print()
print("⚠️  IMPORTANT: After running this cell, please restart the kernel:")
print("   - Click 'Kernel' → 'Restart' in the menu")
print("   - Or use the restart button in the toolbar")
print("   - Then continue running from the next cell")
print()
print("   This ensures the newly installed packages are available.")

📦 Installing required libraries...

   Installing google-cloud-dataplex...



[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


   Installing google-cloud-bigquery...

✅ All libraries installed

⚠️  IMPORTANT: After running this cell, please restart the kernel:
   - Click 'Kernel' → 'Restart' in the menu
   - Or use the restart button in the toolbar
   - Then continue running from the next cell

   This ensures the newly installed packages are available.



[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [36]:
# Initialize clients (run after kernel restart)
import sys

# Force reload the package path if already loaded
if 'google.cloud.dataplex' in sys.modules:
    del sys.modules['google.cloud.dataplex']
if 'google.cloud.dataplex_v1' in sys.modules:
    del sys.modules['google.cloud.dataplex_v1']
if 'google.cloud.bigquery' in sys.modules:
    del sys.modules['google.cloud.bigquery']

try:
    from google.cloud import dataplex_v1
    from google.cloud import bigquery
    
    # Initialize clients
    catalog_client = dataplex_v1.CatalogServiceClient()
    bq_client = bigquery.Client(project=PROJECT_ID)
    
    print("✅ Clients initialized successfully")
    print(f"   Dataplex Catalog Service: Ready")
    print(f"   BigQuery Client: Ready")
    print(f"   Project: {PROJECT_ID}")
    print(f"   Catalog Location: {CATALOG_LOCATION}")
    
except ImportError as e:
    print(f"❌ Import failed: {e}")
    print()
    print("Please restart the kernel and try again:")
    print("   - Click 'Kernel' → 'Restart' in the menu")
    print("   - Run configuration cells again")
    print("   - Skip the install cell (already installed)")
    print("   - Continue from this cell")
    raise

✅ Clients initialized successfully
   Dataplex Catalog Service: Ready
   BigQuery Client: Ready
   Project: johnswain-1200-20260303091357
   Catalog Location: us


---
# Section 2: Analyze Table Schema

Query BigQuery to get all columns from the blockgroup table and categorize them using pattern matching.

In [37]:
# Query BigQuery table schema
print(f"🔍 Analyzing schema for {DATASET_ID}.{TABLE_ID}...")
print()

query = f"""
SELECT 
    column_name,
    data_type
FROM 
    `{PROJECT_ID}.{DATASET_ID}.INFORMATION_SCHEMA.COLUMNS`
WHERE 
    table_name = '{TABLE_ID}'
ORDER BY 
    ordinal_position
"""

try:
    query_job = bq_client.query(query)
    columns = list(query_job.result())
    
    print(f"✅ Found {len(columns)} columns in table {TABLE_ID}")
    print()
    print("📊 Sample columns (first 10):")
    for i, row in enumerate(columns[:10]):
        print(f"   {i+1}. {row.column_name} ({row.data_type})")
    
    print()
    if len(columns) > 10:
        print(f"   ... and {len(columns) - 10} more columns")
    
except Exception as e:
    print(f"❌ Failed to query table schema: {e}")
    raise

🔍 Analyzing schema for census_bureau_acs.blockgroup_2018_5yr...

✅ Found 155 columns in table blockgroup_2018_5yr

📊 Sample columns (first 10):
   1. geo_id (STRING)
   2. do_date (DATE)
   3. total_pop (FLOAT64)
   4. households (FLOAT64)
   5. male_pop (FLOAT64)
   6. female_pop (FLOAT64)
   7. median_age (FLOAT64)
   8. male_under_5 (FLOAT64)
   9. male_5_to_9 (FLOAT64)
   10. male_10_to_14 (FLOAT64)

   ... and 145 more columns


---
# Section 3: Create Business Glossary

Create the root glossary that will contain all categories and terms.

In [38]:
# Define pattern matching rules for categorization
import re

def categorize_column(column_name):
    """
    Categorizes a column based on its name using pattern matching.
    Returns the category ID or None if no match.
    """
    col_lower = column_name.lower()
    
    # Age and Sex patterns
    age_sex_patterns = [
        r'^male_',
        r'^female_',
        r'_age_',
        r'^median_age',
        r'_under_\d+',
        r'_\d+_to_\d+',
        r'^pop_\d+_to_\d+',
        r'^sex_by_age',
        r'_years_',
    ]
    
    # Race and Ethnicity patterns
    race_ethnicity_patterns = [
        r'white_',
        r'black_',
        r'african_american',
        r'asian_',
        r'hispanic_',
        r'latino_',
        r'_race_',
        r'_ethnicity_',
        r'^aian_',  # American Indian/Alaska Native
        r'^nhpi_',  # Native Hawaiian/Pacific Islander
        r'two_or_more_races',
        r'some_other_race',
    ]
    
    # Commuting patterns
    commuting_patterns = [
        r'commute_',
        r'^commute',
        r'_commute',
        r'transportation_',
        r'travel_time',
        r'workers_16',
        r'means_of_transportation',
        r'aggregate_travel_time',
    ]
    
    # Labor Force patterns
    labor_force_patterns = [
        r'labor_force',
        r'employed_',
        r'unemployed_',
        r'^employed',
        r'^unemployed',
        r'_employment',
        r'civilian_labor',
        r'in_labor_force',
        r'not_in_labor_force',
        r'^workers_',
    ]
    
    # Educational Attainment patterns
    education_patterns = [
        r'_degree$',
        r'_diploma$',
        r'high_school',
        r'bachelors',
        r'masters',
        r'doctorate',
        r'_education',
        r'educational_attainment',
        r'associates_degree',
        r'graduate_degree',
        r'less_than_high_school',
    ]
    
    # Check each category
    for pattern in age_sex_patterns:
        if re.search(pattern, col_lower):
            return 'age-and-sex'
    
    for pattern in race_ethnicity_patterns:
        if re.search(pattern, col_lower):
            return 'race-and-ethnicity'
    
    for pattern in commuting_patterns:
        if re.search(pattern, col_lower):
            return 'commuting-characteristics'
    
    for pattern in labor_force_patterns:
        if re.search(pattern, col_lower):
            return 'labor-force'
    
    for pattern in education_patterns:
        if re.search(pattern, col_lower):
            return 'educational-attainment'
    
    return None  # No category match

print("✅ Pattern matching function defined")

✅ Pattern matching function defined


In [39]:
# Categorize all columns
print("🔄 Categorizing columns...")
print()

categorized_columns = {
    'age-and-sex': [],
    'race-and-ethnicity': [],
    'commuting-characteristics': [],
    'labor-force': [],
    'educational-attainment': []
}

uncategorized = []

for row in columns:
    col_name = row.column_name
    col_type = row.data_type
    category = categorize_column(col_name)
    
    if category:
        categorized_columns[category].append({
            'name': col_name,
            'type': col_type
        })
    else:
        uncategorized.append(col_name)

# Display categorization results
print("📊 Categorization Results:")
print()

category_labels = {
    'age-and-sex': 'Age and Sex',
    'race-and-ethnicity': 'Race and Ethnicity',
    'commuting-characteristics': 'Commuting Characteristics',
    'labor-force': 'Labor Force',
    'educational-attainment': 'Educational Attainment'
}

for cat_id, columns_list in categorized_columns.items():
    label = category_labels[cat_id]
    count = len(columns_list)
    print(f"   {label}: {count} columns")
    if count > 0 and count <= 5:
        for col in columns_list:
            print(f"      - {col['name']}")
    elif count > 5:
        for col in columns_list[:3]:
            print(f"      - {col['name']}")
        print(f"      ... and {count - 3} more")

print()
print(f"   Uncategorized: {len(uncategorized)} columns")
if len(uncategorized) > 0 and len(uncategorized) <= 5:
    for col in uncategorized[:5]:
        print(f"      - {col}")
elif len(uncategorized) > 5:
    for col in uncategorized[:3]:
        print(f"      - {col}")
    print(f"      ... and {len(uncategorized) - 3} more")

print()
print(f"✅ Categorization complete")
print(f"   Total categorized: {sum(len(cols) for cols in categorized_columns.values())} columns")
print(f"   Total columns: {len(columns)}")

🔄 Categorizing columns...

📊 Categorization Results:

   Age and Sex: 63 columns
      - male_pop
      - female_pop
      - median_age
      ... and 60 more
   Race and Ethnicity: 7 columns
      - white_pop
      - black_pop
      - asian_pop
      ... and 4 more
   Commuting Characteristics: 12 columns
      - commute_less_10_mins
      - commute_10_14_mins
      - commute_15_19_mins
      ... and 9 more
   Labor Force: 10 columns
      - two_parents_in_labor_force_families_with_young_children
      - two_parents_father_in_labor_force_families_with_young_children
      - two_parents_mother_in_labor_force_families_with_young_children
      ... and 7 more
   Educational Attainment: 4 columns
      - associates_degree
      - bachelors_degree
      - high_school_diploma
      - masters_degree

   Uncategorized: 59 columns
      - geo_id
      - do_date
      - total_pop
      ... and 56 more

✅ Categorization complete
   Total categorized: 96 columns
   Total columns: 155


In [40]:
# Create the root business glossary
print("📚 Creating business glossary...")
print()

# Import required for REST API calls
import requests
from google.auth.transport.requests import Request as AuthRequest

# Get credentials for API calls
creds, _ = google.auth.default()
if not creds.valid:
    creds.refresh(AuthRequest())

# Prepare the REST API request
url = f"https://dataplex.googleapis.com/v1/projects/{PROJECT_ID}/locations/{CATALOG_LOCATION}/glossaries?glossary_id={GLOSSARY_ID}"

glossary_data = {
    "displayName": "ACS Demographics Glossary",
    "description": "Business glossary for US Census Bureau American Community Survey (ACS) demographic data. Implements ISO 11179 Object Classes for organizing demographic concepts including age, sex, race, ethnicity, commuting, labor force, and education."
}

headers = {
    "Authorization": f"Bearer {creds.token}",
    "Content-Type": "application/json"
}

try:
    print("   Creating glossary (this may take a moment)...")
    response = requests.post(url, json=glossary_data, headers=headers)
    
    if response.status_code in [200, 201]:
        result = response.json()
        print(f"✅ Glossary created successfully")
        print(f"   Name: {result.get('name', 'N/A')}")
        print(f"   Display Name: {result.get('displayName', 'N/A')}")
        print(f"   Location: {CATALOG_LOCATION}")
        print()
        print(f"   View in console:")
        print(f"   https://console.cloud.google.com/dataplex/dp-glossaries?project={PROJECT_ID}")
    elif response.status_code == 409:
        print(f"⚠️  Glossary '{GLOSSARY_ID}' already exists")
        print(f"   Continuing with existing glossary...")
        print(f"   To delete and recreate, use the Dataplex console or run:")
        print(f"   gcloud dataplex glossaries delete {GLOSSARY_ID} --location={CATALOG_LOCATION} --project={PROJECT_ID}")
    else:
        print(f"❌ Failed to create glossary")
        print(f"   Status code: {response.status_code}")
        print(f"   Response: {response.text}")
        raise Exception(f"Failed to create glossary: {response.text}")
        
except Exception as e:
    if "already exists" in str(e).lower() or "409" in str(e):
        print(f"⚠️  Glossary '{GLOSSARY_ID}' already exists")
        print(f"   Continuing with existing glossary...")
    else:
        print(f"❌ Error: {e}")
        raise

📚 Creating business glossary...

   Creating glossary (this may take a moment)...
⚠️  Glossary 'acs-demographics-glossary' already exists
   Continuing with existing glossary...
   To delete and recreate, use the Dataplex console or run:
   gcloud dataplex glossaries delete acs-demographics-glossary --location=us --project=johnswain-1200-20260303091357


---
# Section 4: Create Categories (Object Classes)

Create the five main categories representing ISO 11179 Object Classes.

In [41]:
# Define the five categories (Object Classes)
categories_to_create = [
    {
        'id': 'age-and-sex',
        'display_name': 'Age and Sex',
        'description': 'Object Class for demographic data elements related to age distribution and sex/gender characteristics. Includes population counts by age groups and sex, median age statistics, and age-by-sex cross-tabulations.'
    },
    {
        'id': 'race-and-ethnicity',
        'display_name': 'Race and Ethnicity',
        'description': 'Object Class for demographic data elements related to racial and ethnic characteristics. Includes population counts by race categories (White, Black/African American, Asian, American Indian/Alaska Native, Native Hawaiian/Pacific Islander) and Hispanic/Latino ethnicity.'
    },
    {
        'id': 'commuting-characteristics',
        'display_name': 'Commuting Characteristics',
        'description': 'Object Class for data elements related to commuting and transportation patterns. Includes travel time to work, means of transportation, aggregate travel time statistics, and worker mobility characteristics.'
    },
    {
        'id': 'labor-force',
        'display_name': 'Labor Force',
        'description': 'Object Class for data elements related to employment and labor force participation. Includes civilian labor force statistics, employment status, unemployment rates, and labor force participation by demographic groups.'
    },
    {
        'id': 'educational-attainment',
        'display_name': 'Educational Attainment',
        'description': 'Object Class for data elements related to educational achievement levels. Includes high school completion, associate degrees, bachelor degrees, graduate degrees (masters and doctorate), and educational attainment by demographic groups.'
    }
]

print(f"📂 Creating {len(categories_to_create)} categories...")
print()

📂 Creating 5 categories...



In [42]:
# Create each category using REST API
glossary_path = f"projects/{PROJECT_ID}/locations/{CATALOG_LOCATION}/glossaries/{GLOSSARY_ID}"
created_categories = {}
success_count = 0
error_count = 0

for cat_def in categories_to_create:
    cat_id = cat_def['id']
    print(f"   Creating category: {cat_def['display_name']}...")
    
    try:
        # Prepare REST API request
        url = f"https://dataplex.googleapis.com/v1/{glossary_path}/categories?category_id={cat_id}"
        
        category_data = {
            "displayName": cat_def['display_name'],
            "description": cat_def['description'],
            "parent": glossary_path
        }
        
        response = requests.post(url, json=category_data, headers=headers)
        
        if response.status_code in [200, 201]:
            result = response.json()
            created_categories[cat_id] = result.get('name', f"{glossary_path}/categories/{cat_id}")
            success_count += 1
            print(f"      ✅ Created: {cat_id}")
        elif response.status_code == 409:
            print(f"      ⚠️  Already exists: {cat_id}")
            created_categories[cat_id] = f"{glossary_path}/categories/{cat_id}"
            success_count += 1
        else:
            print(f"      ❌ Failed: {response.status_code} - {response.text[:100]}")
            error_count += 1
        
    except Exception as e:
        if "already exists" in str(e).lower() or "409" in str(e):
            print(f"      ⚠️  Already exists: {cat_id}")
            created_categories[cat_id] = f"{glossary_path}/categories/{cat_id}"
            success_count += 1
        else:
            print(f"      ❌ Error: {str(e)[:100]}")
            error_count += 1

print()
print(f"✅ Category creation complete")
print(f"   Success: {success_count}/{len(categories_to_create)}")
if error_count > 0:
    print(f"   Errors: {error_count}")
print()
print(f"   View categories in console:")
print(f"   https://console.cloud.google.com/dataplex/dp-glossaries/{GLOSSARY_ID}?project={PROJECT_ID}")

   Creating category: Age and Sex...
      ❌ Failed: 400 - {
  "error": {
    "code": 400,
    "message": "Category with name: projects/212472948386/locations/
   Creating category: Race and Ethnicity...
      ❌ Failed: 400 - {
  "error": {
    "code": 400,
    "message": "Category with name: projects/212472948386/locations/
   Creating category: Commuting Characteristics...
      ❌ Failed: 400 - {
  "error": {
    "code": 400,
    "message": "Category with name: projects/212472948386/locations/
   Creating category: Labor Force...
      ❌ Failed: 400 - {
  "error": {
    "code": 400,
    "message": "Category with name: projects/212472948386/locations/
   Creating category: Educational Attainment...
      ❌ Failed: 400 - {
  "error": {
    "code": 400,
    "message": "Category with name: projects/212472948386/locations/

✅ Category creation complete
   Success: 0/5
   Errors: 5

   View categories in console:
   https://console.cloud.google.com/dataplex/dp-glossaries/acs-demographics-g

---
# Section 5: Create Glossary Terms

Create glossary terms for each categorized column.

In [43]:
# Helper function to convert column name to display name
def column_to_display_name(column_name):
    """
    Converts a column name to a human-readable display name.
    Example: 'male_35_to_39' -> 'Male Population Age 35 to 39'
    """
    # Replace underscores with spaces
    name = column_name.replace('_', ' ')
    
    # Capitalize each word
    name = ' '.join(word.capitalize() for word in name.split())
    
    # Handle special cases
    name = name.replace(' To ', ' to ')
    name = name.replace(' And ', ' and ')
    name = name.replace(' Or ', ' or ')
    name = name.replace(' Of ', ' of ')
    name = name.replace(' In ', ' in ')
    
    return name

def generate_term_description(column_name, column_type, category_id):
    """
    Generates a description for a glossary term based on the column name and category.
    """
    display = column_to_display_name(column_name)
    
    category_context = {
        'age-and-sex': 'demographic characteristic related to age distribution and sex',
        'race-and-ethnicity': 'demographic characteristic related to racial and ethnic identity',
        'commuting-characteristics': 'characteristic related to commuting patterns and transportation',
        'labor-force': 'characteristic related to employment and labor force participation',
        'educational-attainment': 'characteristic related to educational achievement levels'
    }
    
    context = category_context.get(category_id, 'demographic characteristic')
    
    return f"{display} - A {context}. Data type: {column_type}. Source: US Census Bureau American Community Survey (ACS) blockgroup data."

print("✅ Helper functions defined")

✅ Helper functions defined


In [44]:
# Prepare terms to create (limit to first N terms per category for demo)
MAX_TERMS_PER_CATEGORY = 20  # Configurable: adjust to create more/fewer terms

# VALIDATION: Check that categories actually exist before proceeding
print("🔍 Validating categories exist before creating terms...")
print()

validation_failed = False
for cat_id, cat_path in created_categories.items():
    check_url = f"https://dataplex.googleapis.com/v1/{cat_path}"
    response = requests.get(check_url, headers=headers)
    
    if response.status_code == 200:
        print(f"   ✅ Category exists: {cat_id}")
    else:
        print(f"   ❌ Category NOT found: {cat_id}")
        print(f"      Status: {response.status_code}")
        print(f"      URL: {check_url}")
        validation_failed = True

if validation_failed:
    print()
    print("❌ VALIDATION FAILED: Categories do not exist!")
    print()
    print("   This likely means:")
    print("   1. The categories were not created successfully in Section 4")
    print("   2. You need to re-run Section 4 to create the categories")
    print("   3. Check the glossary exists in the console:")
    print(f"      https://console.cloud.google.com/dataplex/dp-glossaries?project={PROJECT_ID}")
    print()
    raise Exception("Categories validation failed - cannot create terms without valid categories")

print()
print("✅ All categories validated successfully")
print()

# Now prepare terms
terms_to_create = []

for category_id, columns_list in categorized_columns.items():
    # Limit terms per category
    limited_columns = columns_list[:MAX_TERMS_PER_CATEGORY]
    
    for col in limited_columns:
        col_name = col['name']
        col_type = col['type']
        
        terms_to_create.append({
            'column_name': col_name,
            'column_type': col_type,
            'category_id': category_id,
            'category_path': created_categories[category_id],
            'display_name': column_to_display_name(col_name),
            'description': generate_term_description(col_name, col_type, category_id)
        })

print(f"📝 Prepared {len(terms_to_create)} terms to create")
print()
print("   Terms per category:")
for category_id in categorized_columns.keys():
    cat_label = category_labels[category_id]
    count = sum(1 for t in terms_to_create if t['category_id'] == category_id)
    total = len(categorized_columns[category_id])
    print(f"      {cat_label}: {count} of {total} columns")
print()
print(f"   (Limited to {MAX_TERMS_PER_CATEGORY} terms per category for demo)")
print(f"   (Adjust MAX_TERMS_PER_CATEGORY to create more terms)")

🔍 Validating categories exist before creating terms...


✅ All categories validated successfully



KeyError: 'age-and-sex'

In [ ]:
# Create glossary terms using REST API with fail-fast on errors
# NOTE: Terms are created at the GLOSSARY level, not category level
# The category is specified in the term's parent field
print(f"\n🔄 Creating {len(terms_to_create)} glossary terms...")
print("   (This may take several minutes)")
print()

created_terms = {}
term_success = 0
term_errors = 0
MAX_CONSECUTIVE_ERRORS = 3  # Fail fast after 3 consecutive errors
consecutive_errors = 0

# Base URL for creating terms (at glossary level, not category level)
glossary_path = f"projects/{PROJECT_ID}/locations/{CATALOG_LOCATION}/glossaries/{GLOSSARY_ID}"

for i, term_def in enumerate(terms_to_create, 1):
    col_name = term_def['column_name']
    category_path = term_def['category_path']
    
    # Progress indicator
    if i % 10 == 0 or i == 1:
        print(f"   Progress: {i}/{len(terms_to_create)} - Creating '{col_name}'...")
    
    try:
        # Terms are created at the glossary level with parent pointing to category
        # The term ID is passed as 'termId' query parameter (camelCase, not snake_case)
        url = f"https://dataplex.googleapis.com/v1/{glossary_path}/terms?termId={col_name}"
        
        term_data = {
            "displayName": term_def['display_name'],
            "description": term_def['description'],
            "parent": category_path  # This links the term to its category
        }
        
        response = requests.post(url, json=term_data, headers=headers)
        
        if response.status_code in [200, 201]:
            result = response.json()
            created_terms[col_name] = {
                'name': result.get('name', f"{glossary_path}/terms/{col_name}"),
                'category': term_def['category_id']
            }
            term_success += 1
            consecutive_errors = 0  # Reset on success
        elif response.status_code == 409:
            created_terms[col_name] = {
                'name': f"{glossary_path}/terms/{col_name}",
                'category': term_def['category_id']
            }
            term_success += 1
            consecutive_errors = 0  # Reset on success (already exists is OK)
        else:
            term_errors += 1
            consecutive_errors += 1
            
            print(f"      ❌ Error creating '{col_name}':")
            print(f"         Status: {response.status_code}")
            print(f"         URL: {url}")
            
            # Try to parse error message
            try:
                error_json = response.json()
                print(f"         Error: {error_json.get('error', {}).get('message', 'Unknown error')}")
            except:
                print(f"         Response: {response.text[:200]}")
            
            # Fail fast if too many consecutive errors
            if consecutive_errors >= MAX_CONSECUTIVE_ERRORS:
                print()
                print(f"❌ STOPPING: {MAX_CONSECUTIVE_ERRORS} consecutive errors detected")
                print()
                print("   Possible causes:")
                print("   1. Invalid term data or format")
                print("   2. Invalid API credentials or permissions")
                print("   3. API endpoint or format is incorrect")
                print()
                print("   Please check:")
                print(f"   - Glossary in console: https://console.cloud.google.com/dataplex/dp-glossaries?project={PROJECT_ID}")
                print("   - Check IAM permissions (roles/dataplex.catalogAdmin required)")
                raise Exception(f"Failed fast after {MAX_CONSECUTIVE_ERRORS} consecutive errors")
        
    except requests.exceptions.RequestException as e:
        term_errors += 1
        consecutive_errors += 1
        print(f"      ❌ Network error creating '{col_name}': {str(e)[:100]}")
        
        if consecutive_errors >= MAX_CONSECUTIVE_ERRORS:
            raise Exception(f"Failed fast after {MAX_CONSECUTIVE_ERRORS} consecutive network errors")
    
    except Exception as e:
        if "already exists" in str(e).lower() or "409" in str(e):
            created_terms[col_name] = {
                'name': f"{glossary_path}/terms/{col_name}",
                'category': term_def['category_id']
            }
            term_success += 1
            consecutive_errors = 0
        elif "Failed fast" in str(e):
            raise  # Re-raise fail-fast exceptions
        else:
            term_errors += 1
            consecutive_errors += 1
            print(f"      ❌ Exception creating '{col_name}': {str(e)[:100]}")
            
            if consecutive_errors >= MAX_CONSECUTIVE_ERRORS:
                raise Exception(f"Failed fast after {MAX_CONSECUTIVE_ERRORS} consecutive exceptions")

print()
print(f"✅ Term creation complete")
print(f"   Success: {term_success}/{len(terms_to_create)}")
if term_errors > 0:
    print(f"   Errors: {term_errors}")
print()
print(f"   View terms in console:")
print(f"   https://console.cloud.google.com/dataplex/dp-glossaries/{GLOSSARY_ID}?project={PROJECT_ID}")


🔄 Creating 53 glossary terms...
   (This may take several minutes)

   Progress: 1/53 - Creating 'male_pop'...
   Progress: 10/53 - Creating 'male_21'...
   Progress: 20/53 - Creating 'male_62_to_64'...
   Progress: 30/53 - Creating 'commute_15_19_mins'...
   Progress: 40/53 - Creating 'two_parents_in_labor_force_families_with_young_children'...
   Progress: 50/53 - Creating 'associates_degree'...

✅ Term creation complete
   Success: 53/53

   View terms in console:
   https://console.cloud.google.com/dataplex/dp-glossaries/acs-demographics-glossary?project=johnswain-1200-20260303091357


---
# Section 6: Link Terms to Data Assets

Link glossary terms to their corresponding BigQuery table columns.

In [ ]:
# DEBUG: Inspect term structure to find correct field name for linkages
print("🔍 Inspecting term structure to find correct field name...")
print()

import json

# Get a sample term to see its structure
if len(created_terms) > 0:
    sample_term_name = list(created_terms.values())[0]['name']
    inspect_url = f"https://dataplex.googleapis.com/v1/{sample_term_name}"
    inspect_response = requests.get(inspect_url, headers=headers)
    
    if inspect_response.status_code == 200:
        sample_term = inspect_response.json()
        print("✅ Sample term retrieved successfully")
        print()
        print("Available fields in term:")
        for field in sample_term.keys():
            value = sample_term[field]
            print(f"   - {field}: {type(value).__name__} = {str(value)[:100]}")
        print()
        print("Full term structure (JSON):")
        print(json.dumps(sample_term, indent=2))
    else:
        print(f"❌ Failed to retrieve term: {inspect_response.status_code}")
        print(f"   Response: {inspect_response.text[:500]}")
else:
    print("❌ No terms available to inspect")

🔍 Inspecting term structure to find correct field name...

✅ Sample term retrieved successfully

Available fields in term:
   - name: str = projects/johnswain-1200-20260303091357/locations/us/glossaries/acs-demographics-glossary/terms/male_
   - uid: str = 3074e946-3cdd-4b0b-8d03-bafdacd84e93
   - displayName: str = Male Pop
   - description: str = Male Pop - A demographic characteristic related to age distribution and sex. Data type: FLOAT64. Sou
   - createTime: str = 2026-03-07T17:23:36.628639Z
   - updateTime: str = 2026-03-07T17:23:36.628639Z
   - parent: str = projects/johnswain-1200-20260303091357/locations/us/glossaries/acs-demographics-glossary/categories/

Full term structure (JSON):
{
  "name": "projects/johnswain-1200-20260303091357/locations/us/glossaries/acs-demographics-glossary/terms/male_pop",
  "uid": "3074e946-3cdd-4b0b-8d03-bafdacd84e93",
  "displayName": "Male Pop",
  "description": "Male Pop - A demographic characteristic related to age distribution and sex. Data 

In [68]:
# Link glossary terms to BigQuery columns using Dataplex Universal Catalog
# Uses entryLinks API with 'definition' link type to connect terms to columns
print("🔗 Linking glossary terms to BigQuery table columns...")
print()
print("   Strategy: Create entryLinks between glossary terms and table columns")
print()

# BigQuery tables are automatically cataloged in Dataplex Universal Catalog
print(f"   BigQuery Table: {DATASET_ID}.{TABLE_ID}")
print()

try:
    # Step 1: Get project number (required for entry references)
    print("   Step 1: Retrieving project number...")
    project_number_url = f"https://cloudresourcemanager.googleapis.com/v1/projects/{PROJECT_ID}"
    project_response = requests.get(project_number_url, headers=headers, timeout=10)
    
    if project_response.status_code == 200:
        project_number = project_response.json().get('projectNumber')
        print(f"   ✅ Project number: {project_number}")
    else:
        # Fallback: use project ID if we can't get number
        project_number = PROJECT_ID
        print(f"   ⚠️  Could not retrieve project number, using project ID")
    print()
    
    # Step 2: List entries in @bigquery entry group to find our table (with pagination)
    print("   Step 2: Finding BigQuery entry in catalog...")
    import urllib.parse
    
    entry_group_id = "@bigquery"
    fqn = f"bigquery:{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}"
    
    print(f"      Looking for table: {TABLE_ID}")
    print(f"      In dataset: {DATASET_ID}")
    print()
    
    # List entries in the @bigquery entry group with pagination
    print("      Searching through @bigquery entry group...")
    entry_group_path = f"projects/{project_number}/locations/{CATALOG_LOCATION}/entryGroups/{entry_group_id}"
    
    entry_name = None
    entry_found = False
    total_checked = 0
    page_count = 0
    next_page_token = None
    
    while not entry_found and page_count < 20:  # Limit to 20 pages to avoid infinite loop
        page_count += 1
        
        list_url = f"https://dataplex.googleapis.com/v1/{entry_group_path}/entries"
        list_params = {
            "pageSize": 1000  # Get 1000 entries per page
        }
        if next_page_token:
            list_params["pageToken"] = next_page_token
        
        list_response = requests.get(list_url, params=list_params, headers=headers, timeout=30)
        
        if list_response.status_code == 200:
            list_results = list_response.json()
            entries = list_results.get('entries', [])
            next_page_token = list_results.get('nextPageToken')
            
            total_checked += len(entries)
            
            if page_count == 1 or page_count % 5 == 0:
                print(f"      Checked {total_checked} entries so far...")
            
            # Look for our specific table
            for entry in entries:
                entry_fqn = entry.get('fullyQualifiedName', '')
                entry_display = entry.get('entrySource', {}).get('displayName', '')
                
                # Check if this is our table - exact match on table name and dataset
                if entry_display == TABLE_ID and DATASET_ID in entry_fqn:
                    entry_name = entry.get('name')
                    entry_found = True
                    print(f"      ✅ Found BigQuery entry!")
                    print(f"         Entry name: {entry_name}")
                    print(f"         FQN: {entry_fqn}")
                    print(f"         Display: {entry_display}")
                    break
            
            # If no more pages, exit
            if not next_page_token:
                break
        else:
            print(f"      ⚠️  List entries failed: {list_response.status_code}")
            try:
                error_detail = list_response.json()
                error_msg = error_detail.get('error', {}).get('message', 'Unknown')
                print(f"         Error: {error_msg[:300]}")
            except:
                print(f"         Response: {list_response.text[:300]}")
            break
    
    print(f"      Total entries checked: {total_checked}")
    print()
    
    if not entry_found:
        # Check all entries for ones in our dataset
        print(f"      ⚠️  Did not find table '{TABLE_ID}' in dataset '{DATASET_ID}'")
        print()
        print(f"      Let's check what tables are available in the dataset...")
        
        # Do one more pass to find all tables in our dataset
        next_page_token = None
        dataset_entries = []
        page_count = 0
        
        while page_count < 20:
            page_count += 1
            list_url = f"https://dataplex.googleapis.com/v1/{entry_group_path}/entries"
            list_params = {"pageSize": 1000}
            if next_page_token:
                list_params["pageToken"] = next_page_token
            
            list_response = requests.get(list_url, params=list_params, headers=headers, timeout=30)
            if list_response.status_code == 200:
                list_results = list_response.json()
                entries = list_results.get('entries', [])
                
                for entry in entries:
                    entry_fqn = entry.get('fullyQualifiedName', '')
                    if DATASET_ID in entry_fqn:
                        entry_display = entry.get('entrySource', {}).get('displayName', '')
                        dataset_entries.append(entry_display)
                
                next_page_token = list_results.get('nextPageToken')
                if not next_page_token:
                    break
            else:
                break
        
        if dataset_entries:
            print(f"      Found {len(dataset_entries)} tables in dataset '{DATASET_ID}':")
            for i, table_name in enumerate(sorted(set(dataset_entries))[:20], 1):
                print(f"         {i}. {table_name}")
                if table_name == TABLE_ID:
                    print(f"            ^ THIS IS THE ONE WE'RE LOOKING FOR!")
            if len(dataset_entries) > 20:
                print(f"         ... and {len(dataset_entries) - 20} more")
        
        print()
        print(f"      Possible reasons the table wasn't found:")
        print(f"      1. The table exists but isn't cataloged yet (check UI)")
        print(f"      2. Missing IAM permission: roles/dataplex.catalogViewer")
        print(f"      3. The table is in a different location than '{CATALOG_LOCATION}'")
        
        entry_name = None
    
    print()
    
    # Step 3: Verify we can access the entry before trying to link
    if entry_name:
        print("   Step 3: Verifying BigQuery entry access...")
        verify_url = f"https://dataplex.googleapis.com/v1/{entry_name}"
        verify_response = requests.get(verify_url, headers=headers, timeout=10)
        
        if verify_response.status_code == 200:
            print(f"      ✅ Entry accessible")
        else:
            print(f"      ⚠️  Cannot access entry: {verify_response.status_code}")
            print(f"         This may mean:")
            print(f"         1. The BigQuery table hasn't been cataloged yet")
            print(f"         2. You need Dataplex MetadataReader permissions")
            print(f"         3. The entry name format is incorrect")
            print()
            print(f"      Proceeding anyway - links may fail...")
        print()
    
    # Step 4: Create entryLinks for each term-column pair
    print(f"   Step 4: Creating entryLinks for {len(created_terms)} column-term pairs...")
    print("   (This may take a few minutes)")
    print()
    
    if not entry_name:
        print(f"      ❌ Cannot create links: BigQuery entry not found")
        print(f"         Please ensure:")
        print(f"         1. The BigQuery dataset and table exist")
        print(f"         2. You have permissions to access them")
        print(f"         3. The table is in the same location as the catalog ({CATALOG_LOCATION})")
        raise Exception("BigQuery entry not found in catalog")
    
    links_created = []
    link_errors = 0
    consecutive_errors = 0
    MAX_CONSECUTIVE_ERRORS = 3
    
    # Extract entry group from the found entry name for creating links
    # Entry name format: projects/{num}/locations/{loc}/entryGroups/{group}/entries/{entry_id}
    entry_parts = entry_name.split('/entryGroups/')
    if len(entry_parts) == 2:
        entry_group_path = entry_parts[0] + '/entryGroups/' + entry_parts[1].split('/entries/')[0]
    else:
        # Fallback to @bigquery entry group
        entry_group_path = f"projects/{project_number}/locations/{CATALOG_LOCATION}/entryGroups/@bigquery"
    
    for idx, (col_name, term_info) in enumerate(created_terms.items(), 1):
        if idx % 10 == 1 or idx == len(created_terms):
            print(f"      Progress: {idx}/{len(created_terms)}")
        
        try:
            # Each column gets its own entryLink
            # Use hash to keep ID short (max 63 chars), format: term-{hash}-{suffix}
            import hashlib
            col_hash = hashlib.md5(col_name.encode()).hexdigest()[:16]
            entry_link_id = f"term-{col_hash}"
            
            link_url = f"https://dataplex.googleapis.com/v1/{entry_group_path}/entryLinks"
            link_params = {"entryLinkId": entry_link_id}
            
            # Build the term's entry reference
            # Terms are entries in @dataplex entry group
            # The term resource path must also use project_number instead of project_id
            term_resource_path = term_info['name'].replace(f"projects/{PROJECT_ID}", f"projects/{project_number}")
            term_entry_name = f"projects/{project_number}/locations/{CATALOG_LOCATION}/entryGroups/@dataplex/entries/{term_resource_path}"
            
            # DEBUG: Print first link details
            if idx == 1:
                print(f"      DEBUG - First link details:")
                print(f"         BigQuery entry: {entry_name}")
                print(f"         Term entry: {term_entry_name}")
                print(f"         Link URL: {link_url}")
                print()
            
            link_data = {
                "entryLinkType": "projects/dataplex-types/locations/global/entryLinkTypes/definition",
                "entryReferences": [
                    {
                        "name": entry_name,
                        "type": "SOURCE",
                        "path": f"Schema.{col_name}"
                    },
                    {
                        "name": term_entry_name,
                        "type": "TARGET"
                    }
                ]
            }
            
            link_response = requests.post(link_url, json=link_data, headers=headers, params=link_params, timeout=10)
            
            if link_response.status_code in [200, 201]:
                links_created.append({
                    'term': col_name,
                    'column': col_name,
                    'category': term_info['category']
                })
                consecutive_errors = 0
            elif link_response.status_code == 409:
                links_created.append({
                    'term': col_name,
                    'column': col_name,
                    'category': term_info['category']
                })
                consecutive_errors = 0
            else:
                link_errors += 1
                consecutive_errors += 1
                if link_errors <= 5:  # Show first 5 errors
                    print(f"      ⚠️  Error linking '{col_name}': {link_response.status_code}")
                    try:
                        error_msg = link_response.json().get('error', {}).get('message', 'Unknown')
                        print(f"          {error_msg[:500]}")
                    except:
                        pass
                
                # Only stop on consecutive 500-level errors (server issues)
                # 400-level errors are often specific to individual terms
                if link_response.status_code >= 500:
                    if consecutive_errors >= MAX_CONSECUTIVE_ERRORS:
                        print(f"      ❌ Stopping: {MAX_CONSECUTIVE_ERRORS} consecutive server errors")
                        break
                else:
                    # Reset counter for client errors (they're likely different issues)
                    consecutive_errors = 0
        
        except Exception as e:
            link_errors += 1
            consecutive_errors += 1
            if link_errors <= 5:
                print(f"      ⚠️  Exception linking '{col_name}': {str(e)[:100]}")
            if consecutive_errors >= MAX_CONSECUTIVE_ERRORS:
                print(f"      ❌ Stopping: {MAX_CONSECUTIVE_ERRORS} consecutive exceptions")
                break
    
    print()
    link_success = len(links_created)
    
    if link_success > 0:
        print(f"   ✅ Successfully created {link_success} entryLinks")
        if link_errors > 0:
            print(f"   ⚠️  {link_errors} links failed (may already exist or have permission issues)")
    else:
        print(f"   ⚠️  No links created ({link_errors} errors)")

except Exception as e:
    print(f"   ❌ Exception during linking: {str(e)[:200]}")
    link_success = 0
    link_errors = 1
    links_created = []

print()
if link_success > 0:
    print(f"✅ Term linking complete - created {link_success} column-to-term links")
    print()
    print(f"   View linked table in console:")
    print(f"   https://console.cloud.google.com/dataplex/dp-search?q={TABLE_ID}&project={PROJECT_ID}")
else:
    print(f"⚠️  Linking had issues, but glossary is complete:")
    print(f"   - Glossary, 5 categories, {len(created_terms)} terms created")
    print(f"   - You can manually link terms via the Dataplex console")


🔗 Linking glossary terms to BigQuery table columns...

   Strategy: Create entryLinks between glossary terms and table columns

   BigQuery Table: census_bureau_acs.blockgroup_2018_5yr

   Step 1: Retrieving project number...
   ✅ Project number: 212472948386

   Step 2: Finding BigQuery entry in catalog...
      Looking for table: blockgroup_2018_5yr
      In dataset: census_bureau_acs

      Searching through @bigquery entry group...
      Checked 100 entries so far...
      ✅ Found BigQuery entry!
         Entry name: projects/212472948386/locations/us/entryGroups/@bigquery/entries/bigquery.googleapis.com/projects/johnswain-1200-20260303091357/datasets/census_bureau_acs/tables/blockgroup_2018_5yr
         FQN: bigquery:johnswain-1200-20260303091357.census_bureau_acs.blockgroup_2018_5yr
         Display: blockgroup_2018_5yr
      Total entries checked: 343


   Step 3: Verifying BigQuery entry access...
      ✅ Entry accessible

   Step 4: Creating entryLinks for 53 column-term pairs

---
# Section 7: Validation & Exploration

Validate the glossary structure and provide links for manual exploration.

In [69]:
# List all created terms by category
print("📊 Glossary Structure Summary")
print()
print(f"Glossary: ACS Demographics Glossary")
print(f"ID: {GLOSSARY_ID}")
print(f"Location: {CATALOG_LOCATION}")
print()

print("Categories and Terms:")
print()

for category_id, cat_label in category_labels.items():
    # Count terms in this category
    terms_in_cat = [t for t, info in created_terms.items() if info['category'] == category_id]
    print(f"   📂 {cat_label}")
    print(f"      Terms: {len(terms_in_cat)}")
    
    # Show first few terms
    if len(terms_in_cat) > 0:
        print(f"      Sample terms:")
        for term in terms_in_cat[:3]:
            display = column_to_display_name(term)
            print(f"         - {display}")
        if len(terms_in_cat) > 3:
            print(f"         ... and {len(terms_in_cat) - 3} more")
    print()

📊 Glossary Structure Summary

Glossary: ACS Demographics Glossary
ID: acs-demographics-glossary
Location: us

Categories and Terms:

   📂 Age and Sex
      Terms: 20
      Sample terms:
         - Male Pop
         - Female Pop
         - Median Age
         ... and 17 more

   📂 Race and Ethnicity
      Terms: 7
      Sample terms:
         - White Pop
         - Black Pop
         - Asian Pop
         ... and 4 more

   📂 Commuting Characteristics
      Terms: 12
      Sample terms:
         - Commute Less 10 Mins
         - Commute 10 14 Mins
         - Commute 15 19 Mins
         ... and 9 more

   📂 Labor Force
      Terms: 10
      Sample terms:
         - Two Parents in Labor Force Families With Young Children
         - Two Parents Father in Labor Force Families With Young Children
         - Two Parents Mother in Labor Force Families With Young Children
         ... and 7 more

   📂 Educational Attainment
      Terms: 4
      Sample terms:
         - Associates Degree
        

In [70]:
# Validate a sample term with its related entries using REST API
print("🔍 Validating Term Linkages (Sample)")
print()

if len(links_created) > 0:
    sample_link = links_created[0]
    sample_term = sample_link['term']
    term_info = created_terms[sample_term]
    
    print(f"Sample Term: {column_to_display_name(sample_term)}")
    print(f"   ID: {sample_term}")
    print(f"   Category: {category_labels[sample_link['category']]}")
    
    try:
        # Refresh credentials if needed
        from google.auth.transport.requests import Request as AuthRequest
        if not creds.valid:
            creds.refresh(AuthRequest())
        
        fresh_headers = {
            "Authorization": f"Bearer {creds.token}",
            "Content-Type": "application/json"
        }
        
        # Get the term details using REST API
        get_url = f"https://dataplex.googleapis.com/v1/{term_info['name']}"
        get_response = requests.get(get_url, headers=fresh_headers)
        
        if get_response.status_code == 200:
            term = get_response.json()
            
            description = term.get('description', 'No description')
            print(f"   Description: {description[:100]}...")
            print()
            print(f"   ✅ Term is accessible and properly configured")
            
            # Note: Entry links are separate resources, not stored in the term
            print()
            print(f"   💡 To see linked data assets:")
            print(f"      1. Go to Dataplex Universal Catalog")
            print(f"      2. Search for '{sample_term}'")
            print(f"      3. Click on the term to see its links")
            
        else:
            print(f"   ⚠️  Could not retrieve term: {get_response.status_code}")
            if get_response.status_code == 401:
                print(f"      This is an authentication issue (token may have expired)")
                print(f"      The term was created successfully - this validation is optional")
            elif get_response.status_code == 403:
                print(f"      Missing permission to read term metadata")
                print(f"      The term was created successfully - this validation is optional")
        
    except Exception as e:
        print(f"   ⚠️  Could not validate term: {e}")
        print(f"      The term was created successfully - this validation is optional")
else:
    print("   ⚠️  No linked terms to validate")

print()

🔍 Validating Term Linkages (Sample)

Sample Term: Male Pop
   ID: male_pop
   Category: Age and Sex
   ⚠️  Could not retrieve term: 401



In [ ]:
# Provide console links for exploration
print("🌐 Console Links for Exploration")
print()

print("1️⃣  Glossary Overview:")
print(f"   https://console.cloud.google.com/dataplex/dp-glossaries?project={PROJECT_ID}")
print()

print("2️⃣  Specific Glossary (with categories and terms):")
print(f"   https://console.cloud.google.com/dataplex/dp-glossaries/{GLOSSARY_ID}?project={PROJECT_ID}")
print()

print("3️⃣  BigQuery Table:")
print(f"   https://console.cloud.google.com/bigquery?project={PROJECT_ID}&ws=!1m5!1m4!4m3!1s{PROJECT_ID}!2s{DATASET_ID}!3s{TABLE_ID}")
print()

print("4️⃣  Dataplex Catalog Search (search for terms):")
print(f"   https://console.cloud.google.com/dataplex/dp-search?project={PROJECT_ID}")
print()

print("💡 Try these explorations:")
print("   - Browse the glossary to see all categories and terms")
print("   - Click on a term to see its linked data assets")
print("   - Search for a business term (e.g., 'Male Population') in Dataplex Search")
print("   - View the BigQuery table and look for glossary annotations")
print()

---
# Section 8: Summary & Next Steps

Display statistics and provide guidance for extending this implementation.

In [ ]:
# Display comprehensive statistics
print("="*70)
print("📈 GLOSSARY IMPLEMENTATION SUMMARY")
print("="*70)
print()

print("✅ Successfully Created:")
print()
print(f"   📚 Glossary: 1")
print(f"      Name: ACS Demographics Glossary")
print(f"      ID: {GLOSSARY_ID}")
print(f"      Location: {CATALOG_LOCATION}")
print()

print(f"   📂 Categories (Object Classes): {len(created_categories)}")
for cat_id, cat_label in category_labels.items():
    print(f"      - {cat_label}")
print()

print(f"   📝 Glossary Terms: {len(created_terms)}")
for category_id, cat_label in category_labels.items():
    terms_count = sum(1 for t, info in created_terms.items() if info['category'] == category_id)
    total_cols = len(categorized_columns[category_id])
    print(f"      - {cat_label}: {terms_count} terms (from {total_cols} columns)")
print()

print(f"   🔗 Term-to-Column Linkages: {link_success}")
print(f"      Each term is linked to its corresponding BigQuery column")
print()

print(f"📊 Data Coverage:")
print(f"   Target Table: {PROJECT_ID}.{DATASET_ID}.{TABLE_ID}")
print(f"   Total Columns in Table: {len(columns)}")
total_categorized = sum(len(cols) for cols in categorized_columns.values())
print(f"   Columns Categorized: {total_categorized}")
print(f"   Terms Created: {len(created_terms)}")
coverage_pct = (len(created_terms) / len(columns) * 100) if len(columns) > 0 else 0
print(f"   Coverage: {coverage_pct:.1f}%")
print()

print("="*70)

In [ ]:
# Provide next steps and guidance
print("\n🚀 NEXT STEPS & GUIDANCE")
print("="*70)
print()

print("1️⃣  Explore Your Glossary:")
print("   - Visit the Dataplex console to browse categories and terms")
print("   - Click on terms to see their linked data assets")
print("   - Try searching for business terms in Dataplex Search")
print()

print("2️⃣  Extend to More Tables:")
print("   To apply glossary terms to other blockgroup tables:")
print("   - Update TABLE_ID variable (e.g., 'blockgroup_2017_5yr')")
print("   - Re-run Sections 2, 5, and 6")
print("   - Terms will be linked to additional tables")
print()

print("3️⃣  Create More Terms:")
print("   To include more columns:")
print("   - Increase MAX_TERMS_PER_CATEGORY in Section 5")
print("   - Re-run Sections 5 and 6")
print("   - All categorized columns can have terms")
print()

print("4️⃣  Enhance Pattern Matching:")
print("   To categorize more columns:")
print("   - Add patterns in Section 2's categorize_column() function")
print("   - Consider adding new categories for housing, income, etc.")
print("   - Re-run from Section 2 onwards")
print()

print("5️⃣  Add Term Relationships:")
print("   Enrich your glossary with:")
print("   - Synonyms (e.g., 'Male Pop' synonym for 'Male Population')")
print("   - Related terms (e.g., link 'Employed' with 'Labor Force')")
print("   - Use the Dataplex console or API to add these relationships")
print()

print("6️⃣  Maintain the Glossary:")
print("   - Regularly review term descriptions for accuracy")
print("   - Update categories as your data evolves")
print("   - Add contacts (stewards/owners) to categories and terms")
print("   - Consider adding overview documentation to categories")
print()

print("7️⃣  Integrate with Data Governance:")
print("   - Use glossary terms in data quality rules")
print("   - Reference terms in documentation and lineage")
print("   - Train data users on the glossary terminology")
print("   - Establish a process for term approval and updates")
print()

print("="*70)
print()
print("✨ Your ISO 11179-compliant glossary is ready for use!")
print()
print(f"   Main Console Link:")
print(f"   https://console.cloud.google.com/dataplex/dp-glossaries/{GLOSSARY_ID}?project={PROJECT_ID}")
print()

In [ ]:
# Optional: Export glossary structure to JSON for documentation
import json
from datetime import datetime

glossary_export = {
    'metadata': {
        'glossary_id': GLOSSARY_ID,
        'glossary_name': 'ACS Demographics Glossary',
        'location': CATALOG_LOCATION,
        'project': PROJECT_ID,
        'created_date': datetime.now().isoformat(),
        'target_table': f"{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}"
    },
    'categories': {},
    'statistics': {
        'total_categories': len(created_categories),
        'total_terms': len(created_terms),
        'total_linkages': link_success,
        'table_columns': len(columns),
        'coverage_percent': round(coverage_pct, 1)
    }
}

# Add categories and their terms
for category_id, cat_label in category_labels.items():
    terms_in_cat = [t for t, info in created_terms.items() if info['category'] == category_id]
    glossary_export['categories'][category_id] = {
        'display_name': cat_label,
        'term_count': len(terms_in_cat),
        'terms': terms_in_cat
    }

# Pretty print the export
export_json = json.dumps(glossary_export, indent=2)

print("📄 Glossary Structure (JSON Export)")
print()
print("This JSON can be saved for documentation purposes:")
print()
print(export_json[:500] + "...")
print()
print(f"   Full export length: {len(export_json)} characters")
print()
print("   To save to file, uncomment and run:")
print("   # with open('glossary_export.json', 'w') as f:")
print("   #     f.write(export_json)")